# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Logistic Regression, then Random Forest** (Gradient Boosting tried as a stretch). My lane's question is "which pages first?" — a ranking question, built on top of `is_declining` (a binary observed proxy, verified leakage-free in ML-04). Per the toolkit: a yes/no observed label starts with Logistic Regression (readable), escalating to Random Forest only if the extra complexity earns it.

Evaluated the same way as the ML-07 baseline: **precision@K**, because the decision is capacity-constrained ("review the top K first"), not "classify everyone correctly."

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# find the repo root from wherever this kernel started (VS Code/Colab/CLI all differ)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

# Token order: env var -> Colab Secret -> prompt (last resort). Never hardcode — repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_timeout=300")  # seconds; default is 30s, too short for the heavier queries below
con.execute("SET http_retries=5")
con.execute("SET http_retry_wait_ms=1000")
con.execute("SET http_retry_backoff=2")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# same feature build + proxy label verified in ML-04, same rows the ML-07 baseline scored
feat = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_prev,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS active_days_prev,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last
    FROM read_parquet('{MONTH}')
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0
""").df()

feat["ctr_prev"] = feat["clk_prev"] / feat["imp_prev"]
feat["is_declining"] = (feat["imp_last"] < 0.8 * feat["imp_prev"]).astype(int)
feat = feat.dropna(subset=["avg_position_prev"]).reset_index(drop=True)

# log-transform the two heavy-tailed volume columns for the linear model; trees don't need it
feat["log_imp_prev"] = np.log1p(feat["imp_prev"])
feat["log_clk_prev"] = np.log1p(feat["clk_prev"])

FEATURES = ["log_imp_prev", "log_clk_prev", "ctr_prev", "avg_position_prev", "active_days_prev"]
print(f"shape: {feat.shape[0]:,} rows  |  clients: {feat['client_hash_id'].nunique()}  |  base decline rate: {feat['is_declining'].mean() * 100:.1f}%")

shape: 150,675 rows  |  clients: 44  |  base decline rate: 32.6%


## 2. Split design

**Grouped by `client_hash_id`** (`GroupShuffleSplit`, 25% of clients held out, `random_state=42`) — not a random row split. Content items from the same client share client-level quirks (traffic scale, industry, tracking setup); a random row split would let the model see other rows from the same client during training and quietly memorize the client instead of the pattern, inflating precision@K for the wrong reason. `client_hash_id` is a pseudonym per the ML-04 data contract — grouping only, never a feature.

Verified below: zero client overlap between train and test.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

X = feat[FEATURES]
y = feat["is_declining"]
groups = feat["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = feat.iloc[test_idx].copy()

train_clients = set(feat.iloc[train_idx]["client_hash_id"])
test_clients = set(feat.iloc[test_idx]["client_hash_id"])

print(f"train: {len(train_idx):,} rows, {len(train_clients)} clients")
print(f"test:  {len(test_idx):,} rows, {len(test_clients)} clients")
print(f"train base rate: {y_train.mean():.3f}  |  test base rate: {y_test.mean():.3f}")

assert train_clients.isdisjoint(test_clients), "LEAKAGE: a client appears in both train and test"
print("assert passed: zero client overlap between train and test.")

train: 136,249 rows, 33 clients
test:  14,426 rows, 11 clients
train base rate: 0.322  |  test base rate: 0.366
assert passed: zero client overlap between train and test.


## 3. Train + compare vs my baseline

The ML-07 baseline rule, recomputed on this exact test split (same data, same metric, same split — not the whole-dataset number from ML-07). Table below.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_train, y_train)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42).fit(X_train, y_train)

test_df["p_logreg"] = logreg.predict_proba(X_test_s)[:, 1]
test_df["p_rf"] = rf.predict_proba(X_test)[:, 1]
test_df["p_gb"] = gb.predict_proba(X_test)[:, 1]

# ML-07 baseline rule, recomputed on THIS test set only (thresholds fit on train, applied to test)
train_eligible = (X_train["avg_position_prev"] > 0) & (X_train["avg_position_prev"] <= 20) & (feat.iloc[train_idx]["imp_prev"] >= 100)
tier = pd.cut(X_train.loc[train_eligible, "avg_position_prev"], bins=[0, 3, 10, 20], labels=["1-3", "4-10", "11-20"]).astype(str)
expected_ctr_by_tier = X_train.loc[train_eligible].assign(position_tier=tier).groupby("position_tier")["ctr_prev"].median().to_dict()

def expected_ctr_for_position(pos):
    if pos <= 3:
        return expected_ctr_by_tier["1-3"]
    if pos <= 10:
        return expected_ctr_by_tier["4-10"]
    return expected_ctr_by_tier["11-20"]

test_df["expected_ctr"] = test_df["avg_position_prev"].apply(expected_ctr_for_position)
test_eligible = (test_df["imp_prev"] >= 100) & (test_df["avg_position_prev"] > 0) & (test_df["avg_position_prev"] <= 20)
gap = np.where(test_eligible, test_df["expected_ctr"] - test_df["ctr_prev"], 0.0)
test_df["score_baseline"] = np.where(test_eligible, test_df["imp_prev"] * np.clip(gap, 0, None), 0.0)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

Ks = [10, 20, 50, 100]
rows = []
for name, col in [("baseline (ML-07 rule)", "score_baseline"), ("logistic_regression", "p_logreg"),
                   ("random_forest", "p_rf"), ("gradient_boosting", "p_gb")]:
    row = {"method": name}
    for k in Ks:
        row[f"precision@{k}"] = round(precision_at_k(test_df[col], test_df["is_declining"], k), 3)
    rows.append(row)

comparison = pd.DataFrame(rows)
print(f"test base rate: {test_df['is_declining'].mean():.3f}  (test rows: {len(test_df):,}, test clients: {test_df['client_hash_id'].nunique()})")
comparison

test base rate: 0.366  (test rows: 14,426, test clients: 11)


,method,precision@10,precision@20,precision@50,precision@100
0,baseline (ML-07 rule),0.5,0.60,0.56,0.54
1,logistic_regression,0.4,0.45,0.40,0.42
2,random_forest,0.2,0.45,0.42,0.47
3,gradient_boosting,0.5,0.50,0.50,0.42


## 4. Errors and interpretation

**Honest finding: on this grouped test split, none of the three models beat the hand-written baseline at any K.** The baseline wins or ties every column; Gradient Boosting only ties at precision@10 and falls behind from precision@20 on; Random Forest and Logistic Regression trail throughout. Random Forest's own test ROC-AUC sits around 0.58 — barely above a coin flip. Adding model complexity did not earn its keep here: five basic features and only 11 held-out clients aren't enough to out-rank a rule built on the same two confirmed signals. That's the "don't reward complexity alone" result the assignment asks me to report honestly, not paper over.

**What the model leans on (permutation importance, Random Forest, scored on the held-out test set):** `log_imp_prev` (volume) first, then `ctr_prev`, `active_days_prev`, `avg_position_prev`, with `log_clk_prev` last — matches the two signals CONFIRMED in ML-07 (volume, CTR-vs-position). The top importance is small (well under a suspicious near-1.0 AUC swing), consistent with a real but modest signal — not leakage.

**Three concrete wrong cases** (see below): all three false positives are very-high-volume pages sitting at a very poor position (38, 39, 56) that the model reads as high-risk purely from volume + bad rank — but none of them declined, plausibly because they're already at a floor with nowhere lower to fall. The false negatives are the mirror image: pages with an excellent position (~1.4–2.5) and real volume that still declined — good position isn't protective on its own, and the model under-weights that.

In [5]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

print(f"random_forest test ROC-AUC: {roc_auc_score(y_test, test_df['p_rf']):.3f}")

perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42, scoring="roc_auc", n_jobs=-1)
print("\npermutation importance (drop in test ROC-AUC when a feature is shuffled):")
for feat_name, imp, std in sorted(zip(FEATURES, perm.importances_mean, perm.importances_std), key=lambda t: -t[1]):
    print(f"  {feat_name:20s} {imp:.4f} +/- {std:.4f}")

show_cols = FEATURES + ["p_rf", "is_declining"]
by_score = test_df.sort_values("p_rf", ascending=False)

print("\n--- false positives (high predicted risk, did NOT decline) ---")
print(by_score[by_score["is_declining"] == 0].head(3)[show_cols].to_string())

print("\n--- false negatives (low predicted risk, DID decline) ---")
print(by_score[by_score["is_declining"] == 1].tail(3)[show_cols].to_string())

random_forest test ROC-AUC: 0.578

permutation importance (drop in test ROC-AUC when a feature is shuffled):
  log_imp_prev         0.0381 +/- 0.0041
  ctr_prev             0.0277 +/- 0.0020
  active_days_prev     0.0249 +/- 0.0027
  avg_position_prev    0.0205 +/- 0.0021
  log_clk_prev         0.0062 +/- 0.0013

--- false positives (high predicted risk, did NOT decline) ---
        log_imp_prev  log_clk_prev  ctr_prev  avg_position_prev  active_days_prev      p_rf  is_declining
78409      10.128350      2.079442  0.000280          38.271078                15  0.849476             0
118700      9.418898      2.197225  0.000649          38.718105                15  0.831364             0
135171      8.846785      2.197225  0.001151          55.996136                15  0.817028             0

--- false negatives (low predicted risk, DID decline) ---
        log_imp_prev  log_clk_prev  ctr_prev  avg_position_prev  active_days_prev      p_rf  is_declining
46073       7.847372      3.25809

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.